# EDA RAG Docs

Exploratory analysis for RAG documents.

Steps:
- Inventory document folders and extensions.
- Preview a small text sample.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from collections import Counter
from pathlib import Path

rag_dirs = [
    REPO_ROOT / 'data' / 'docs',
    REPO_ROOT / 'data' / 'samples' / 'docs',
]

summary = {
    'rag_dirs': [],
    'extensions': {},
    'sample_files': [],
}

for rag_dir in rag_dirs:
    print('RAG dir:', rag_dir)
    if not rag_dir.exists():
        print('Missing:', rag_dir)
        continue
    files = [p for p in rag_dir.rglob('*') if p.is_file()]
    summary['rag_dirs'].append({'path': str(rag_dir), 'file_count': len(files)})
    ext_counts = Counter(p.suffix.lower() or 'no_ext' for p in files)
    summary['extensions'][str(rag_dir)] = dict(ext_counts.most_common(8))
    print('Files:', len(files))
    print('Top extensions:', dict(ext_counts.most_common(6)))
    summary['sample_files'].extend([str(p.relative_to(REPO_ROOT)) for p in files[:5]])


In [ ]:
# Preview a small text document if available.
text_file = None
for path_str in summary['sample_files']:
    path = REPO_ROOT / path_str
    if path.suffix.lower() in {'.md', '.txt'}:
        text_file = path
        break

if text_file and text_file.exists():
    print('Preview:', text_file.relative_to(REPO_ROOT))
    print(text_file.read_text(encoding='utf-8', errors='ignore')[:2000])
else:
    print('No text document found for preview.')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_rag_docs_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize rag-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'rag' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No rag entries found in TRAINING_DATA.json')
    else:
        print('rag datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
